# Hospital Dataset Cleaning

This notebook cleans the hospital raw dataset and creates a Tableau-ready dataset.

### Cleaning Operations
1. Remove duplicate records
2. Handle missing patient data
3. Standardize department names
4. Normalize healthcare indicators
5. Standardize date fields
6. Export Tableau-ready CSV


In [ ]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv('hospital_raw_dataset.csv')
print('Raw dataset shape:', df.shape)
df.head()


In [ ]:
# Remove duplicate records
duplicates = df.duplicated().sum()
df = df.drop_duplicates().reset_index(drop=True)
print('Duplicates removed:', duplicates)


In [ ]:
# Handle missing values
for col in df.columns:
    if df[col].isna().sum() > 0:
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col] = df[col].fillna(df[col].median())
        else:
            mode = df[col].mode()
            df[col] = df[col].fillna(
                mode.iloc[0] if len(mode) else 'Unknown'
            )


In [ ]:
# Standardize department names
department_mapping = {
    'cardio': 'Cardiology',
    'cardiology dept': 'Cardiology',
    'ortho': 'Orthopedics',
    'orthopaedics': 'Orthopedics',
    'ent': 'ENT',
    'emergency dept': 'Emergency',
    'general med': 'General Medicine',
    'neuro': 'Neurology',
    'gastro': 'Gastroenterology',
    'paediatrics': 'Pediatrics'
}

for col in df.columns:
    if 'department' in col or col in ['dept', 'unit']:
        df[col] = df[col].apply(
            lambda x: department_mapping.get(
                str(x).strip().lower(),
                x
            ) if pd.notna(x) else x
        )


In [ ]:
# Normalize healthcare indicators
indicator_mapping = {
    'yes': 'Yes', 'y': 'Yes', 'true': 'Yes',
    'no': 'No', 'n': 'No', 'false': 'No',
    'male': 'Male', 'm': 'Male',
    'female': 'Female', 'f': 'Female',
    'positive': 'Positive', 'pos': 'Positive',
    'negative': 'Negative', 'neg': 'Negative'
}

for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].apply(
        lambda x: indicator_mapping.get(
            str(x).strip().lower(), x
        ) if pd.notna(x) else x
    )


In [ ]:
# Save Tableau-ready dataset
df.to_csv('hospital_cleaned.csv', index=False)

print('hospital_cleaned.csv created successfully!')
print('Final shape:', df.shape)
print('Missing values:', df.isna().sum().sum())
